# REVE Subject-Specific Prompts — Two-Stage Fine-Tuning

Two-stage training strategy:
1. **Stage 1:** Train only the classification head (prompt tokens frozen) — the head learns a good mapping from pooled features to classes
2. **Stage 2:** Train only the subject prompt tokens (classifier frozen) — the prompts specialize per-subject while the classifier stays stable

## Loading REVE

In [1]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True, torch_dtype="auto")
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True, torch_dtype="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


flash_attn not found, install it with `pip install flash_attn` if you want to use it


## Subject-Specific Prompt Wrapper

In [2]:
import torch.nn as nn
from einops import rearrange


class ReveSubjectPrompt(nn.Module):
    def __init__(self, reve_model, num_subject, num_prompt=1, num_classes=4, dropout=0.1):
        super().__init__()
        self.reve = reve_model
        self.embed_dim = reve_model.embed_dim
        self.num_subject = num_subject
        self.num_prompt = num_prompt

        # Freeze the backbone
        for param in self.reve.parameters():
            param.requires_grad = False

        # Subject-specific prompt tokens: one set per subject
        self.subject_prompt_tokens = nn.ParameterList([
            nn.Parameter(torch.randn(1, num_prompt, self.embed_dim))
            for _ in range(num_subject)
        ])

        # Classification head: num_prompt * embed_dim -> num_classes
        self.classifier = nn.Sequential(
            nn.RMSNorm(num_prompt * self.embed_dim),
            nn.Dropout(dropout),
            nn.Linear(num_prompt * self.embed_dim, num_classes),
        )

    def freeze_prompts(self):
        """Freeze prompt tokens, unfreeze classifier."""
        for p in self.subject_prompt_tokens:
            p.requires_grad = False
        for p in self.classifier.parameters():
            p.requires_grad = True

    def freeze_classifier(self):
        """Freeze classifier, unfreeze prompt tokens."""
        for p in self.subject_prompt_tokens:
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = False

    def unfreeze_all(self):
        """Unfreeze both prompts and classifier."""
        for p in self.subject_prompt_tokens:
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True

    def subject_attention_pooling(self, x, subject_ids):
        b, c, s, e = x.shape
        x = rearrange(x, "b c s e -> b (c s) e")

        queries = torch.stack([
            self.subject_prompt_tokens[sid.item()] for sid in subject_ids
        ]).squeeze(1)

        attention_scores = torch.matmul(queries, x.transpose(-1, -2)) / (self.embed_dim ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        out = torch.matmul(attention_weights, x)

        return out.reshape(b, -1)

    def forward(self, eeg, pos, subject_ids):
        with torch.no_grad():
            x = self.reve(eeg, pos)

        pooled = self.subject_attention_pooling(x, subject_ids)
        return self.classifier(pooled)

In [3]:
# Training parameters
num_subject = 9
num_prompt = 4
batch_size = 64
n_epochs_stage1 = 10  # classifier head only
n_epochs_stage2 = 10  # prompt tokens only
lr_stage1 = 1e-3
lr_stage2 = 1e-3
positions = pos_bank(["Fz", "FC3", "FC1", "FCz", "FC2", "FC4", "C5", "C3", "C1", "Cz", "C2", "C4", "C6", "CP3", "CP1", "CPz", "CP2", "CP4", "P1", "Pz", "P2", "POz"])

prompt_model = ReveSubjectPrompt(model, num_subject=num_subject, num_prompt=num_prompt, num_classes=4)
print(f"Total trainable parameters: {sum(p.numel() for p in prompt_model.parameters() if p.requires_grad):,}")
print(f"  - Subject prompt tokens: {num_subject} x ({num_prompt} x {model.embed_dim}) = {num_subject * num_prompt * model.embed_dim:,}")
print(f"  - Classifier parameters: {sum(p.numel() for p in prompt_model.classifier.parameters()):,}")

Total trainable parameters: 28,676
  - Subject prompt tokens: 9 x (4 x 512) = 18,432
  - Classifier parameters: 10,244


In [4]:
from transformers import set_seed

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Data Loading

In [5]:
from functools import partial
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from scipy.signal import butter, lfilter
import numpy as np
from torch.utils.data import Dataset, random_split

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype="band")

paradigm = MotorImagery(n_classes=4, resample=250, fmin=8, fmax=30)
bci_dataset = BNCI2014_001()
X, y, metadata = paradigm.get_data(dataset=bci_dataset)

b, a = butter_bandpass(8, 30, 250)
X = lfilter(b, a, X, axis=-1)

label_map = {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3}
y = np.array([label_map[label] for label in y])

unique_subjects = sorted(metadata["subject"].unique())
subject_to_idx = {s: i for i, s in enumerate(unique_subjects)}
subject_ids = np.array([subject_to_idx[s] for s in metadata["subject"]])
print(f"Subjects: {unique_subjects} -> indices {list(subject_to_idx.values())}")


class BCIDatasetWithSubject(Dataset):
    def __init__(self, X, y, subject_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.subject_ids = torch.tensor(subject_ids, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {"data": self.X[idx], "labels": self.y[idx], "subject_id": self.subject_ids[idx]}


n = len(y)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val
full_dataset = BCIDatasetWithSubject(X, y, subject_ids)
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])


def collate(batch, positions):
    x_data = torch.stack([x["data"] for x in batch])
    y_label = torch.tensor([x["labels"] for x in batch])
    sid = torch.tensor([x["subject_id"] for x in batch])
    positions = positions.repeat(len(batch), 1, 1)
    return {"sample": x_data, "label": y_label.long(), "pos": positions, "subject_id": sid}


collate_fn = partial(collate, positions=positions)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

Choosing from all possible events


Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)] -> indices [0, 1, 2, 3, 4, 5, 6, 7, 8]


## Training Functions

In [6]:
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize


def train_one_epoch(model, optimizer, loader):
    model.train()
    pbar = tqdm(loader, desc="Training", total=len(loader))

    for batch_data in pbar:
        data, target, pos, subject_id = (
            batch_data["sample"].to(device, non_blocking=True),
            batch_data["label"].to(device, non_blocking=True),
            batch_data["pos"].to(device, non_blocking=True),
            batch_data["subject_id"].to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        with torch.amp.autocast(dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"):
            output = model(data, pos, subject_id)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})


def eval_model(model, loader):
    model.eval()

    y_decisions = []
    y_targets = []
    y_probs = []
    score, count = 0, 0
    pbar = tqdm(loader, desc="Evaluating", total=len(loader))
    with torch.inference_mode():
        for batch_data in pbar:
            data, target, pos, subject_id = (
                batch_data["sample"].to(device, non_blocking=True),
                batch_data["label"].to(device, non_blocking=True),
                batch_data["pos"].to(device, non_blocking=True),
                batch_data["subject_id"].to(device, non_blocking=True),
            )
            with torch.amp.autocast(
                dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"
            ):
                output = model(data, pos, subject_id)

            decisions = torch.argmax(output, dim=1)
            score += (decisions == target).int().sum().item()
            count += target.shape[0]
            y_decisions.append(decisions)
            y_targets.append(target)
            y_probs.append(output)

    gt = torch.cat(y_targets).cpu().numpy()
    pr = torch.cat(y_decisions).cpu().numpy()
    pr_probs = torch.softmax(torch.cat(y_probs).float(), dim=1).cpu().numpy()
    acc = score / count
    balanced_acc = balanced_accuracy_score(gt, pr)
    cohen_kappa = cohen_kappa_score(gt, pr)
    f1 = f1_score(gt, pr, average="weighted")

    auroc = roc_auc_score(gt, pr_probs, multi_class='ovr')
    auc_pr = average_precision_score(label_binarize(gt, classes=[0, 1, 2, 3]), pr_probs, average='macro')

    return acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr

## Stage 1: Train Classification Head Only

Prompt tokens are frozen — the classifier learns to map the (random) pooled representations to the 4 classes.

In [7]:
criterion = torch.nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
prompt_model.to(device)

# Stage 1: freeze prompts, train classifier only
prompt_model.freeze_prompts()
trainable_stage1 = [p for p in prompt_model.parameters() if p.requires_grad]
print(f"Stage 1 — trainable params: {sum(p.numel() for p in trainable_stage1):,} (classifier only)")

optimizer1 = torch.optim.AdamW(trainable_stage1, lr=lr_stage1)
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer1, mode="max", factor=0.5, patience=3)

best_val_acc = 0
best_model_state = None

for epoch in range(n_epochs_stage1):
    print(f"[Stage 1] Epoch {epoch + 1}/{n_epochs_stage1}")
    train_one_epoch(prompt_model, optimizer1, train_loader)
    _, b_acc, _, _, _, _ = eval_model(prompt_model, val_loader)
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_model_state = {k: v.clone() for k, v in prompt_model.state_dict().items()}
    print(f"  Val balanced acc: {b_acc:.4f}, best: {best_val_acc:.4f}")
    scheduler1.step(b_acc)

# Reload best stage 1 model
prompt_model.load_state_dict(best_model_state)
print(f"\nStage 1 complete — best val balanced acc: {best_val_acc:.4f}")

Stage 1 — trainable params: 10,244 (classifier only)
[Stage 1] Epoch 1/10


Training:   0%|          | 0/57 [00:00<?, ?it/s]

/users/local/miniconda3/envs/venv/lib/python3.12/site-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3223, best: 0.3223
[Stage 1] Epoch 2/10


...

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3519, best: 0.3538
[Stage 1] Epoch 10/10


Training:   0%|          | 0/57 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3477, best: 0.3538

Stage 1 complete — best val balanced acc: 0.3538


## Stage 2: Train Subject Prompt Tokens Only

The classifier is now frozen. Each subject's prompt tokens learn to produce the best attention queries for that subject.

In [8]:
# Stage 2: freeze classifier, train prompts only
prompt_model.freeze_classifier()
trainable_stage2 = [p for p in prompt_model.parameters() if p.requires_grad]
print(f"Stage 2 — trainable params: {sum(p.numel() for p in trainable_stage2):,} (prompts only)")

optimizer2 = torch.optim.AdamW(trainable_stage2, lr=lr_stage2)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode="max", factor=0.5, patience=3)

best_val_acc = 0
best_model_state = None

for epoch in range(n_epochs_stage2):
    print(f"[Stage 2] Epoch {epoch + 1}/{n_epochs_stage2}")
    train_one_epoch(prompt_model, optimizer2, train_loader)
    _, b_acc, _, _, _, _ = eval_model(prompt_model, val_loader)
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_model_state = {k: v.clone() for k, v in prompt_model.state_dict().items()}
    print(f"  Val balanced acc: {b_acc:.4f}, best: {best_val_acc:.4f}")
    scheduler2.step(b_acc)

# Reload best stage 2 model
prompt_model.load_state_dict(best_model_state)
print(f"\nStage 2 complete — best val balanced acc: {best_val_acc:.4f}")

Stage 2 — trainable params: 18,432 (prompts only)
[Stage 2] Epoch 1/10


Training:   0%|          | 0/57 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3487, best: 0.3487
[Stage 2] Epoch 2/10


Training:   0%|          | 0/57 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3514, best: 0.3542
[Stage 2] Epoch 10/10


Training:   0%|          | 0/57 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Val balanced acc: 0.3502, best: 0.3542

Stage 2 complete — best val balanced acc: 0.3542


## Test Evaluation

In [9]:
acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr = eval_model(prompt_model, test_loader)

print("\n--- Test Results (Two-Stage) ---")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Cohen's Kappa:     {cohen_kappa:.4f}")
print(f"F1 (weighted):     {f1:.4f}")
print(f"AUROC:             {auroc:.4f}")
print(f"AUC-PR:            {auc_pr:.4f}")

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]


--- Test Results (Two-Stage) ---
Accuracy:          0.3145
Balanced Accuracy: 0.3111
Cohen's Kappa:     0.0816
F1 (weighted):     0.3116
AUROC:             0.5874
AUC-PR:            0.3184
